In [28]:

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import pandas as pd
import matplotlib.pyplot as plt
import time
import numpy as np

In [2]:
url_official_voting_record = 'https://www.theyworkforyou.com/mps/'


In [125]:
#Importing a data frame that contains links to the voting record of every British mp
MP_voting_record_url = pd.read_csv("mps.csv")
MP_voting_record_url.head()

,Person ID,First name,Last name,Party,Constituency,URI
0,10001,Diane,Abbott,Independent,Hackney North and Stoke Newington,https://www.theyworkforyou.com/mp/10001/diane_...
1,26385,Jack,Abbott,Labour/Co-operative,Ipswich,https://www.theyworkforyou.com/mp/26385/jack_a...
2,25034,Debbie,Abrahams,Labour,Oldham East and Saddleworth,https://www.theyworkforyou.com/mp/25034/debbie...
3,26364,Shockat,Adam,Independent,Leicester South,https://www.theyworkforyou.com/mp/26364/shocka...
4,26483,Zubir,Ahmed,Labour,Glasgow South West,https://www.theyworkforyou.com/mp/26483/zubir_...


In [120]:
#Validating urls in the dataframe
MP_voting_record_url.iloc[50,5]

'https://www.theyworkforyou.com/mp/25726/orfhlaith_begley/west_tyrone'

In [45]:
#Converting url column to a list
url_list = MP_voting_record_url["URI"].to_list()

    

In [3]:
#Function that takes one of the urls from the list as argument
#then access the site with selenium, click through to voting record summary 
#and returns the text of voting history converted from html to text format
def get_text(url):
    driver = webdriver.Chrome()
    driver.get(url)
    element = driver.find_element("link text", "Voting record summary").click()
    records = driver.find_elements(By.CLASS_NAME, "vote-description")


    text = [r.text for r in records]
    splitted = [t.split("\n")[0] for t in text]
    driver.close()
    return splitted



In [ ]:
#list comprehension that iterates through the list of urls applying each to the created function: get_text()
final_record = [get_text(link) for link in inc_url_list]

In [127]:
#dropping the list of urls from the orignal dataframe
new_df = MP_voting_record_url.drop("URI", axis=1)

In [129]:
#creating a new column to contain the obtained voting record 
new_df["Voting Record"] = final_record

In [130]:
#validating new column and dataframe
new_df

,Person ID,First name,Last name,Party,Constituency,Voting Record
0,10001,Diane,Abbott,Independent,Hackney North and Stoke Newington,[Almost always voted against a reduction in sp...
1,26385,Jack,Abbott,Labour/Co-operative,Ipswich,[Consistently voted for measures to prevent cl...
2,25034,Debbie,Abrahams,Labour,Oldham East and Saddleworth,"[Generally voted for smoking bans, Consistentl..."
3,26364,Shockat,Adam,Independent,Leicester South,[Consistently voted against allowing terminall...
4,26483,Zubir,Ahmed,Labour,Glasgow South West,"[Voted for smoking bans, Consistently voted ag..."
...,...,...,...,...,...,...
645,26600,Yuan,Yang,Labour,Earley and Woodley,[Voted for means-testing/removing universality...
646,25649,Mohammad,Yasin,Labour,Bedford,[Consistently voted against allowing terminall...
647,26469,Steve,Yemm,Labour,Mansfield,"[Voted for smoking bans, Consistently voted fo..."
648,26458,Claire,Young,Liberal Democrat,Thornbury and Yate,[Consistently voted for more rights for tenant...


In [131]:
#Saving work to a csv to process later
new_df.to_csv("mp_voting_record.csv")

In [6]:
#downloading the csv containing register of interests / donors and donations to MP's
donations = pd.read_csv("overall.csv")

In [7]:
donations

,id,parent_interest_id,member,party,mnis_id,twfy_id,category,category_code,summary,__index_level_0__
0,495,NaN,Jon Trickett,labour,410,10604,Donations and other support (including loans) ...,2.0,"Communications Workers' Union - £2,000.00",534
1,818,NaN,Dame Siobhain McDonagh,labour,193,10381,"Gifts, benefits and hospitality from UK sources",3.0,"Waheed Alli - £1,200,000.00",581
2,1049,NaN,Ed Davey,liberal-democrat,188,10155,"Gifts, benefits and hospitality from UK sources",3.0,National Liberal Club - £798.00,580
3,1117,NaN,Tim Farron,liberal-democrat,1591,11923,"Gifts, benefits and hospitality from UK sources",3.0,"National Liberal Club - £1,102.50",579
4,1193,NaN,Ruth Cadbury,labour,4389,25343,Family members engaged in third-party lobbying,10.0,Nicholas Gash employed as Lobbyist,15
...,...,...,...,...,...,...,...,...,...,...
3805,13459,NaN,Laura Kyrke-Smith,labour,5341,26629,Visits outside the UK,4.0,International visit to United States between 1...,0
3806,13460,NaN,Richard Baker,labour,5253,13952,Visits outside the UK,4.0,International visit to Ukraine between 10 Sept...,3
3807,13461,NaN,Mark Sewards,labour,5166,26440,Visits outside the UK,4.0,International visit to Ukraine between 10 Sept...,2
3808,13463,NaN,Bridget Phillipson,labour,4046,24709,Miscellaneous,8.0,I am the beneficiary of the work of Bridget Ph...,1


In [8]:
#data validation
print(donations["party"].value_counts())
print(donations.info())


party
labour                                1472
conservative                          1160
liberal-democrat                       537
independent                            185
labourco-operative                     159
reform                                 153
scottish-national-party                 39
green                                   29
plaid-cymru                             27
sinn-fein                               20
dup                                     15
traditional-unionist-voice               5
uup                                      4
unknown                                  3
social-democratic-and-labour-party       2
Name: count, dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3810 entries, 0 to 3809
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  3810 non-null   int64  
 1   parent_interest_id  643 non-null    float64
 2   member         

In [ ]:
#Text processing on summary column

In [109]:
#Splitting summary into text and amount columns
donations[[ "summarys", "amount"]] = donations["summary"].str.split("-", expand=True, n=1)

In [ ]:
#Several steps to reformat amount column to have just the numerical data to allow for calculations in later stages

In [110]:
donations["amount"] = donations["amount"].str.replace("£" , "")

In [111]:
donations["amount"] = donations["amount"].str.strip()

In [112]:
donations["amount"] = donations["amount"].str.replace("." , ",")

In [113]:
donations["amount"] = donations["amount"].str.replace("," , "")

In [125]:
donations["amount"] = donations["amount"].str.replace("" , "0")

In [99]:
donations["amount"] = donations["amount"].fillna(0)

In [128]:
donations["amount"] = donations["amount"].str.strip()

In [114]:
#Using string regex to remove eronius characters in amount column
donations[['summary2', 'amount']] = donations['amount'].str.extract(r'([\d.]+)(\w+)')

In [118]:
donations["summary2"] = donations["summary2"].str[:-1]

In [120]:
#Replacing missing values with 0, to keep data usable 
donations["summary2"] = donations["summary2"].fillna(0)

In [158]:
#Replacing empty values with 0
donations["summary2"] = np.where(donations["summary2"] == "", 0, donations["summary2"])


In [161]:
#changing data type of donations amounts to integer
donations["summary2"] = donations["summary2"].astype("int")


In [164]:
donations2 = donations[["member", "party", "category", "category_code", "summary", "summarys", "summary2"]]

In [ ]:
#changing names of new columns 
donations2.rename(columns={"summary2": "amount", "summarys" : "summary_text"},inplace=True)

In [168]:
donations2

,member,party,category,category_code,summary,summary_text,amount
0,Jon Trickett,labour,Donations and other support (including loans) ...,2.0,"Communications Workers' Union - £2,000.00",Communications Workers' Union,2000
1,Dame Siobhain McDonagh,labour,"Gifts, benefits and hospitality from UK sources",3.0,"Waheed Alli - £1,200,000.00",Waheed Alli,1200000
2,Ed Davey,liberal-democrat,"Gifts, benefits and hospitality from UK sources",3.0,National Liberal Club - £798.00,National Liberal Club,798
3,Tim Farron,liberal-democrat,"Gifts, benefits and hospitality from UK sources",3.0,"National Liberal Club - £1,102.50",National Liberal Club,1102
4,Ruth Cadbury,labour,Family members engaged in third-party lobbying,10.0,Nicholas Gash employed as Lobbyist,Nicholas Gash employed as Lobbyist,0
...,...,...,...,...,...,...,...
3805,Laura Kyrke-Smith,labour,Visits outside the UK,4.0,International visit to United States between 1...,International visit to United States between 1...,0
3806,Richard Baker,labour,Visits outside the UK,4.0,International visit to Ukraine between 10 Sept...,International visit to Ukraine between 10 Sept...,0
3807,Mark Sewards,labour,Visits outside the UK,4.0,International visit to Ukraine between 10 Sept...,International visit to Ukraine between 10 Sept...,0
3808,Bridget Phillipson,labour,Miscellaneous,8.0,I am the beneficiary of the work of Bridget Ph...,I am the beneficiary of the work of Bridget Ph...,0


In [169]:
#saving work to csv
donations2.to_csv("mp_donations_processed.csv")

In [2]:
mp_donors = pd.read_csv("mp_donations_processed.csv")

In [171]:
mp_donors.sort_values("amount",ascending=False)

,member,party,category,category_code,summary,summary_text,amount
1,Dame Siobhain McDonagh,labour,"Gifts, benefits and hospitality from UK sources",3.0,"Waheed Alli - £1,200,000.00",Waheed Alli,1200000
3006,Charlie Maynard,liberal-democrat,"Gifts, benefits and hospitality from UK sources",3.0,"William Day - £301,800.00",William Day,301800
614,Sir Geoffrey Cox,conservative,Employment and earnings - Ongoing paid employment,1.2,"Agreement starting 01 November 2023 - £293,400.00",Agreement starting 01 November 2023,293400
3005,Charlie Maynard,liberal-democrat,"Gifts, benefits and hospitality from UK sources",3.0,"Marriott Harrison LLP - £282,000.00",Marriott Harrison LLP,282000
3186,David Davis,conservative,Donations and other support (including loans) ...,2.0,"Trailfinders Ltd - £250,000.00",Trailfinders Ltd,250000
...,...,...,...,...,...,...,...
1355,Manuela Perteghella,liberal-democrat,Miscellaneous,8.0,"Member of executive group, Stratford upon Avon...","Member of executive group, Stratford upon Avon...",0
2412,Naz Shah,labour,Visits outside the UK,4.0,International visit to Qatar between 06 Decemb...,International visit to Qatar between 06 Decemb...,0
2411,Nick Timothy,conservative,Miscellaneous,8.0,Trustee of the Index on Censorship. This is an...,Trustee of the Index on Censorship. This is an...,0
2410,Melanie Ward,labour,Visits outside the UK,4.0,International visit to Qatar between 06 Decemb...,International visit to Qatar between 06 Decemb...,0


In [172]:
mp_donors["category"].value_counts()

category
Miscellaneous                                                            909
Gifts, benefits and hospitality from UK sources                          582
Donations and other support (including loans) for activities as an MP    536
Employment and earnings - Ad hoc payments                                535
Employment and earnings                                                  338
Visits outside the UK                                                    317
Land and property (within or outside the UK)                             216
Shareholdings                                                            200
Employment and earnings - Ongoing paid employment                        108
Family members employed                                                   34
Family members engaged in third-party lobbying                            22
Gifts and benefits from sources outside the UK                            13
Name: count, dtype: int64

In [5]:
#Removing unnecessary columns
mp_donors = mp_donors[["member", "category", "summary_text", "amount"]]

In [9]:
#checking for duplicated rows
mp_donors[mp_donors.duplicated()]

,member,category,summary_text,amount
245,Rushanara Ali,Land and property (within or outside the UK),Residential in London,0
325,Saqib Bhatti,Land and property (within or outside the UK),Residential in Walsall,0
389,Kevin Hollinrake,Land and property (within or outside the UK),Residential in York,0
407,Sir Desmond Swayne,Land and property (within or outside the UK),Residential in London,0
464,Sir Iain Duncan Smith,Employment and earnings,Writing articles,0
...,...,...,...,...
3739,Sir Keir Starmer,"Gifts, benefits and hospitality from UK sources",The Arsenal Football Club Limited,1000
3752,Charlie Maynard,"Gifts, benefits and hospitality from UK sources",William Day,4200
3772,Steff Aquarone,Donations and other support (including loans) ...,Anthony Smith,2500
3798,Nigel Farage,Employment and earnings,Speaking engagement,0


In [13]:
mp_donors[mp_donors["member"] == "Nigel Farage"]

,member,category,summary_text,amount
1063,Nigel Farage,Employment and earnings,Presenter,0
1064,Nigel Farage,Employment and earnings - Ad hoc payments,Payment received on 30 September 2024,60388
1065,Nigel Farage,Employment and earnings - Ad hoc payments,Payment received on 30 October 2024,35433
1066,Nigel Farage,Employment and earnings - Ad hoc payments,Payment received on 29 November 2024,42076
1067,Nigel Farage,Employment and earnings - Ad hoc payments,Payment received on 15 January 2025,45286
...,...,...,...,...
3799,Nigel Farage,Employment and earnings - Ad hoc payments,Payment received on 08 May 2025,6493
3800,Nigel Farage,Employment and earnings - Ad hoc payments,Payment received on 19 May 2025,6493
3801,Nigel Farage,Employment and earnings - Ad hoc payments,Payment received on 10 June 2025,12986
3802,Nigel Farage,Employment and earnings,Speaking engagement,0


In [18]:
#checking for missing values
mp_donors["member"].isna().any()

False

In [14]:
#saving final mp donor data to csv
mp_donors.to_csv("mp_donors.csv")

In [64]:
voting_record = pd.read_csv("mp_voting_record.csv")

In [ ]:
#Splitting the list of MP's voting record to seperate values

In [212]:
voting_record["Voting Record"] = voting_record["Voting Record"].str.split(",")

In [8]:
voting_record["Voting Record"] = voting_record["Voting Record"].str.split(",")

In [37]:
#turning voting record into a list of lists
list_of_lists = voting_record["Voting Record"].to_list()

In [9]:
#Converting MP first and last name to full name
voting_record["Full_Name"] = voting_record["First name"] + " " + voting_record["Last name"]

In [10]:
#Dropping first and last name
voting_record.drop(["First name", "Last name"], axis="columns", inplace=True)

In [11]:
#creating a list of MP names to process with list of votes
names = voting_record["Full_Name"].to_list()

In [109]:
#dataframe to join MP name and MP vote 
vote_per_row = pd.DataFrame({"name" : ["Diane Abbott"], "vote" : ["test vote string"]})

In [110]:
vote_per_row

,name,vote
0,Diane Abbott,test vote string


In [111]:
#iterating through list of MP names and list of MP votes to add each vote to the dataframe 
for name in names:
    list_of_votes = voting_record[voting_record["Full_Name"] == name]["Voting Record"].to_list()[0]
    for vote in list_of_votes:
        vote_per_row.loc[len(vote_per_row)] = [name, vote]
        

In [112]:
vote_per_row

,name,vote
0,Diane Abbott,test vote string
1,Diane Abbott,['Almost always voted against a reduction in s...
2,Diane Abbott,'Generally voted against reducing housing ben...
3,Diane Abbott,'Almost always voted for paying higher benefi...
4,Diane Abbott,'Generally voted for raising welfare benefits...
...,...,...
37092,Daniel Zeichner,'Consistently voted for improving biodiversity'
37093,Daniel Zeichner,'Consistently voted for improving air quality'
37094,Daniel Zeichner,'Consistently voted for improving environment...
37095,Daniel Zeichner,'Consistently voted for increasing windfall t...


In [113]:
vote_per_row = vote_per_row[1:]

In [ ]:
#Text formatting on the created dataframe of MP votes
vote_per_row["vote"] = vote_per_row["vote"].str.replace("[", "")
vote_per_row["vote"] = vote_per_row["vote"].str.replace("]", "")
vote_per_row["vote"] = vote_per_row["vote"].str.replace("'", "")
vote_per_row["vote"] = vote_per_row["vote"].str.replace('"', "")
vote_per_row["vote"] = np.where(vote_per_row["vote"].str.contains(")", regex=False), vote_per_row["vote"].str.split(")")[1], vote_per_row["vote"])

In [ ]:
vote_per_row["vote"] = vote_per_row["vote"].str.strip()

In [121]:
df.to_csv("votes_sorted.csv")

In [14]:
sorted = pd.read_csv("votes_sorted.csv")

In [17]:
#Creating a dictionary mapping mp sentiment to a numerical score to allow for numerical and visual comparisons of MP's votes
sentiment_dict = {"Almost always voted for" : "11", "Consistently voted for" : "10", "Voted for" : "9", "Generally voted for" : "8", 
                  "Tended to vote for" : "7", "Voted a mixture of for and against" : "6", "Tended to vote against" : "5", "Generally voted against" : "4",
                 "Voted against" : "3", "Consistently voted against" : "2", "Almost always voted against" : "1"}

sorted["vote"].replace(sentiment_dict,inplace=True, regex=True)

In [19]:
#splitting vote column from eronious numerical data 
sorted[["sentiment_number", "vote"]] = sorted["vote"].str.split(r'([\d.]+)', expand=True,n=1).drop(0, axis=1)

In [21]:
sorted

,Unnamed: 0,name,vote,sentiment_number
0,3,Diane Abbott,paying higher benefits over longer periods fo...,11
1,12,Diane Abbott,measures to prevent climate change,11
2,56,Diane Abbott,more powers for the devolved administration i...,11
3,108,Diane Abbott,equal gay rights,11
4,111,Diane Abbott,laws to promote equality and human rights,11
...,...,...,...,...
34873,37053,Daniel Zeichner,a reduction in spending on welfare benefits,1
34874,37072,Daniel Zeichner,a stricter asylum system,1
34875,37074,Daniel Zeichner,stronger laws and enforcement of immigration ...,1
34876,37089,Daniel Zeichner,a reduction in spending on welfare benefits,1


In [22]:
#Dropping uneeded column
sorted = sorted.drop(columns="Unnamed: 0")

In [23]:
#converting sentiment score to integer
sorted["sentiment_number"] = sorted["sentiment_number"].astype("int")

In [24]:
#removing duplicates
sorted = sorted.drop_duplicates(keep="first")

In [25]:
sorted.to_csv("mp_votes_sentiment.csv")

In [26]:
mp_votes = pd.read_csv("mp_votes_sentiment.csv")

In [27]:
mp_votes

,Unnamed: 0,name,vote,sentiment_number
0,0,Diane Abbott,paying higher benefits over longer periods fo...,11
1,1,Diane Abbott,measures to prevent climate change,11
2,2,Diane Abbott,more powers for the devolved administration i...,11
3,3,Diane Abbott,equal gay rights,11
4,4,Diane Abbott,laws to promote equality and human rights,11
...,...,...,...,...
28256,34863,Steve Yemm,a reduction in spending on welfare benefits,1
28257,34865,Claire Young,a reduction in spending on welfare benefits,1
28258,34866,Daniel Zeichner,a reduction in spending on welfare benefits,1
28259,34874,Daniel Zeichner,a stricter asylum system,1
